In [3]:
from pymoo.core.problem import Problem
import numpy as np

class QAPProblem(Problem):
    def __init__(self, A, B):
        self.A = A
        self.B = B
        n = A.shape[0]

        super().__init__(
            n_var=n, # number of variable
            n_obj=1, # objective functions to be minimized, 1 since we minimize cost
            n_constr=0, # constraints
            xl=0, # lower bound
            xu=n-1, # upper bound
            type_var=int
        )
    def _evaluate(self, X, out, *args, **kwargs):
        # fill F with 0's / initialization
        F = np.zeros(X.shape[0], dtype=np.int64)
        for k in range(X.shape[0]):
            pi = X[k].astype(int) # get permutation
            # cost calculation
            F[k] = np.sum(self.A * self.B[np.ix_(pi, pi)])
        out["F"] = F

In [4]:
from pymoo.core.sampling import Sampling
from pymoo.core.crossover import Crossover
from pymoo.core.mutation import Mutation

class PermutationRandomSampling(Sampling):
    def _do(self, problem, n_samples, **kwargs):
        n = problem.n_var
        X = np.zeros((n_samples, n), dtype=int)
        base = np.arange(n)
        for i in range(n_samples):
            X[i] = np.random.permutation(base)
        return X

class OrderCrossover(Crossover):
    def __init__(self, prob=0.9):
        super().__init__(2, 2, prob=prob)

    def _do(self, problem, X, **kwargs):
        n_matings, _, n = X.shape
        Y = np.full((n_matings, 2, n), -1, dtype=int)

        for m in range(n_matings):
            p1 = X[m, 0].astype(int)
            p2 = X[m, 1].astype(int)

            a, b = sorted(np.random.choice(n, 2, replace=False))

            c1 = np.full(n, -1, dtype=int)
            c1[a:b+1] = p1[a:b+1]
            fill = [g for g in p2 if g not in c1]
            idx = [i for i in range(n) if c1[i] == -1]
            c1[idx] = fill

            c2 = np.full(n, -1, dtype=int)
            c2[a:b+1] = p2[a:b+1]
            fill = [g for g in p1 if g not in c2]
            idx = [i for i in range(n) if c2[i] == -1]
            c2[idx] = fill

            Y[m, 0, :] = c1
            Y[m, 1, :] = c2

        return Y

class SwapMutation(Mutation):
    def __init__(self, prob=0.2):
        super().__init__()
        self.prob = prob

    def _do(self, problem, X, **kwargs):
        X = X.copy().astype(int)
        n_samples, n = X.shape
        for i in range(n_samples):
            if np.random.rand() < self.prob:
                a, b = np.random.choice(n, 2, replace=False)
                X[i, a], X[i, b] = X[i, b], X[i, a]
        return X

In [6]:
from pymoo.algorithms.soo.nonconvex.ga import GA
from pymoo.optimize import minimize
from pymoo.termination import get_termination



algorithm = GA(
    pop_size=300,
    sampling=PermutationRandomSampling(),
    crossover=OrderCrossover(prob=0.9),
    mutation=SwapMutation(prob=0.2),
    eliminate_duplicates=True
)

res = minimize(
    problem,
    algorithm,
    get_termination("n_gen", 50000),
    seed=1,
    verbose=True
)

NameError: name 'problem' is not defined